In [1]:
import pandas as pd


In [2]:
import pandas as pd
import numpy as np

import re
import string
from nltk.stem import WordNetLemmatizer

from nltk.corpus import stopwords

from sklearn.model_selection import (
    train_test_split,
    RandomizedSearchCV,
    StratifiedKFold
)

from sklearn.preprocessing import (
    LabelEncoder,
    OneHotEncoder,
    StandardScaler
)

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from scipy.sparse import hstack

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import warnings
warnings.filterwarnings("ignore")

In [3]:
df=pd.read_csv('../data/customer_support_tickets.csv')

In [4]:
selected_columns = [
    'Ticket Subject',
    'Ticket Description',
    'Customer Age',
    'Customer Gender',
    'Product Purchased',
    'Ticket Channel',
    'Ticket Type',
    'Ticket Priority'
]

df = df[selected_columns]

In [5]:
# ============================================================
# 4. HANDLE MISSING VALUES
# ============================================================

# TEXT COLUMNS
text_cols = ['Ticket Subject', 'Ticket Description']

for col in text_cols:
    df[col] = df[col].fillna("missing")

# CATEGORICAL COLUMNS
cat_cols = [
    'Customer Gender',
    'Product Purchased',
    'Ticket Channel',
    'Ticket Type'
]

for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

# NUMERIC COLUMNS
num_cols = [
    'Customer Age'
]

for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

In [6]:
# ============================================================
# 5. TEXT PREPROCESSING
# ============================================================

def clean_text(text):
    lemmatizer= WordNetLemmatizer()
    stop_words=set(stopwords.words('english'))
    # Lowercase
    text = text.lower()

    # Remove numbers
    text = re.sub(r'\d+', '', text)

    # # Remove punctuation
    # text = text.translate(
    #     str.maketrans('', '', string.punctuation)
    # )

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text
    # # Tokenization
    # words = text.split()

    # # Remove stopwords + lemmatization
    # cleaned_words = []

    # for word in words:

    #     # Remove stopwords
    #     if word not in stop_words:

    #         # Lemmatize
    #         lemma_word = lemmatizer.lemmatize(word)

    #         cleaned_words.append(lemma_word)

    # # Join back
    # text = " ".join(cleaned_words)

    # return text


df['clean_subject'] = df['Ticket Subject'].apply(clean_text)

df['clean_description'] = df['Ticket Description'].apply(clean_text)


# ============================================================
# 6. COMBINE TEXT
# ============================================================

df['combined_text'] = (
    df['clean_subject'] + " " +
    df['clean_description']
)

In [7]:
# ============================================================
# 7. FEATURE ENGINEERING
# ============================================================

# CHARACTER COUNT
df['char_count'] = df['combined_text'].apply(len)

# WORD COUNT
df['word_count'] = df['combined_text'].apply(
    lambda x: len(x.split())
)

# AVERAGE WORD LENGTH
df['avg_word_length'] = df['combined_text'].apply(
    lambda x: np.mean([len(word) for word in x.split()])
    if len(x.split()) > 0 else 0
)

# EXCLAMATION COUNT
df['exclamation_count'] = (
    df['Ticket Description']
    .astype(str)
    .apply(lambda x: x.count('!'))
)

# QUESTION COUNT
df['question_count'] = (
    df['Ticket Description']
    .astype(str)
    .apply(lambda x: x.count('?'))
)


In [8]:
# ============================================================
# 8. ENCODE TARGET VARIABLE
# ============================================================
target_map={
    'Critical':3,
    'High':2,
    'Medium':1,
    'Low':0
}
df['target']=df['Ticket Priority'].map(target_map)


In [9]:
df.head()

,Ticket Subject,Ticket Description,Customer Age,Customer Gender,Product Purchased,Ticket Channel,Ticket Type,Ticket Priority,clean_subject,clean_description,combined_text,char_count,word_count,avg_word_length,exclamation_count,question_count,target
0,Product setup,I'm having an issue with the {product_purchase...,32,Other,GoPro Hero,Social media,Technical issue,Critical,product setup,i'm having an issue with the {product_purchase...,product setup i'm having an issue with the {pr...,290,45,5.466667,0,0,3
1,Peripheral compatibility,I'm having an issue with the {product_purchase...,42,Female,LG Smart TV,Chat,Technical issue,Critical,peripheral compatibility,i'm having an issue with the {product_purchase...,peripheral compatibility i'm having an issue w...,304,46,5.630435,0,0,3
2,Network problem,I'm facing a problem with my {product_purchase...,48,Other,Dell XPS,Social media,Technical issue,Low,network problem,i'm facing a problem with my {product_purchase...,network problem i'm facing a problem with my {...,287,44,5.545455,0,0,0
3,Account access,I'm having an issue with the {product_purchase...,27,Female,Microsoft Office,Social media,Billing inquiry,Low,account access,i'm having an issue with the {product_purchase...,account access i'm having an issue with the {p...,276,43,5.441860,0,0,0
4,Data loss,I'm having an issue with the {product_purchase...,67,Female,Autodesk AutoCAD,Email,Billing inquiry,Low,data loss,i'm having an issue with the {product_purchase...,data loss i'm having an issue with the {produc...,341,57,5.000000,0,0,0


In [10]:
# ============================================================
# 9. DEFINE FEATURES
# ============================================================

categorical_features = [
    'Customer Gender',
    'Product Purchased',
    'Ticket Channel',
    'Ticket Type'
]

numerical_features = [
    'Customer Age',
    'char_count',
    'word_count',
    'avg_word_length',
    'exclamation_count',
    'question_count'
]

text_feature = 'combined_text'

In [11]:
# ============================================================
# 10. TF-IDF VECTORIZATION
# ============================================================

tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2)
)

X_text = tfidf.fit_transform(df[text_feature])

In [12]:
import nltk

# Download the specific tokenizer model resource required by word_tokenize
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/rishabh/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [13]:
import gensim.downloader as api
from nltk.tokenize import word_tokenize
import nltk

nltk.download('punkt')

[nltk_data] Downloading package punkt to /Users/rishabh/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [14]:
# ==========================================
# 1. LOAD PRE-TRAINED GLOVE EMBEDDINGS
# ==========================================
print("Loading GloVe embeddings (this may take a minute)...")
# 'glove-wiki-gigaword-100' creates a 100-dimensional vector for each word
glove_model = api.load("glove-wiki-gigaword-100")

Loading GloVe embeddings (this may take a minute)...


In [15]:
# ==========================================
# 2. DEFINE TEXT-TO-VECTOR FUNCTION
# ==========================================
def get_document_embedding(text, model, vector_size=100):
    if not isinstance(text, str) or text.strip() == "":
        return np.zeros(vector_size)
    
    # Tokenize words and convert to lowercase
    words = [word.lower() for word in word_tokenize(text)]
    
    # Extract vectors only for words that exist in the GloVe vocabulary
    valid_vectors = [model[word] for word in words if word in model]
    
    # If no words match the vocabulary, return a vector of zeros
    if not valid_vectors:
        return np.zeros(vector_size)
    
    # Return the average (mean) vector of the text
    return np.mean(valid_vectors, axis=0)

In [16]:
# Assuming your text column is named 'Ticket Description'
print("Converting text column into GloVe dense vectors...")
text_vectors = df['combined_text'].apply(lambda x: get_document_embedding(x, glove_model, 100))

Converting text column into GloVe dense vectors...


In [17]:
glove_df = pd.DataFrame(text_vectors.tolist(), columns=[f'glove_{i}' for i in range(100)])

In [18]:
# ============================================================
# 11. TABULAR PREPROCESSING
# ============================================================

preprocessor = ColumnTransformer(
    transformers=[

        (
            'cat',
            OneHotEncoder(handle_unknown='ignore'),
            categorical_features
        ),

        (
            'num',
            StandardScaler(),
            numerical_features
        )
    ]
)

X_tabular = preprocessor.fit_transform(
    df[categorical_features + numerical_features]
)

In [20]:
X_tabular.shape

(8469, 60)

In [21]:
glove_df.shape

(8469, 100)

In [23]:
# ============================================================
# 12. COMBINE TF-IDF + TABULAR FEATURES
# ============================================================

X = hstack([glove_df, X_tabular])

y = df['target']


# ============================================================
# 13. TRAIN TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [24]:
xgb_model = XGBClassifier(
    objective='multi:softprob',
    num_class=4,
    eval_metric='mlogloss',
    random_state=42
)

xgb_model.fit(X_train, y_train)
y_pred = xgb_model.predict(X_test)

In [25]:
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='weighted')
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')

In [26]:
# Print out metrics dashboard
print("=" * 50)
print("        XGBOOST CLASSIFIER METRICS REPORT        ")
print("=" * 50)
print(f"Accuracy  : {accuracy:.4f}")
print(f"F1-Score  : {f1:.4f}  (Weighted)")
print(f"Precision : {precision:.4f}  (Weighted)")
print(f"Recall    : {recall:.4f}  (Weighted)")
print("-" * 50)

        XGBOOST CLASSIFIER METRICS REPORT        
Accuracy  : 0.2379
F1-Score  : 0.2380  (Weighted)
Precision : 0.2385  (Weighted)
Recall    : 0.2379  (Weighted)
--------------------------------------------------


In [ ]:
from sklearn.dummy import DummyClassifier

dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train, y_train)
print(f"Random Guess Baseline: {dummy.score(X_test, y_test):.4f}")

Random Guess Baseline: 0.2586
